# Week 3: NumPy & Pandas for Biology

**Goal:** Master the two libraries that make Python a real data analysis platform. After this week, you'll handle data faster than you ever could in Excel.

By the end of this notebook, you will:
- Use NumPy arrays for fast numerical operations on biology data
- Load datasets into Pandas DataFrames
- Filter, group, merge, and summarize biology datasets
- Handle missing values in real-world data
- Work with the Breast Cancer Wisconsin dataset

**Time:** ~2-3 hours | **Difficulty:** Beginner-Intermediate

**Prerequisites:** `pip install numpy pandas scikit-learn`

---

## 1. NumPy — Fast Math on Biology Data

NumPy arrays are like Python lists, but optimized for numbers. Operations that take seconds on lists take milliseconds on arrays. This matters when you have 20,000 genes across 500 samples.

In [ ]:
import numpy as np

# Creating arrays from biology data
expression = np.array([5.2, 12.4, 3.1, 8.7, 22.1, 4.8, 6.7, 3.2])
gene_names = ["TP53", "EGFR", "BRCA1", "KRAS", "MYC", "PTEN", "PIK3CA", "BRAF"]

print(f"Expression array: {expression}")
print(f"Type: {type(expression)}")
print(f"Shape: {expression.shape}")
print(f"Data type: {expression.dtype}")

In [ ]:
# NumPy statistics — one function call each!
print("Expression Statistics:")
print(f"  Mean:   {np.mean(expression):.2f}")
print(f"  Median: {np.median(expression):.2f}")
print(f"  Std:    {np.std(expression):.2f}")
print(f"  Min:    {np.min(expression):.2f} ({gene_names[np.argmin(expression)]})")
print(f"  Max:    {np.max(expression):.2f} ({gene_names[np.argmax(expression)]})")
print(f"  Sum:    {np.sum(expression):.2f}")

In [ ]:
# Vectorized operations — apply math to ALL values at once
# No loops needed! This is what makes NumPy fast.

# Log2 transform (common in gene expression analysis)
log2_expression = np.log2(expression)
print("Log2 transformed:")
for gene, raw, log in zip(gene_names, expression, log2_expression):
    print(f"  {gene:<8} {raw:>6.1f}  →  {log:>5.2f}")

# Z-score normalization (how many std deviations from the mean)
z_scores = (expression - np.mean(expression)) / np.std(expression)
print(f"\nZ-scores: {np.round(z_scores, 2)}")

In [ ]:
# Boolean indexing — filter with conditions
# This is MUCH faster than loops for large datasets

# Which genes are highly expressed (> 7)?
high_mask = expression > 7.0
print(f"Mask: {high_mask}")
print(f"Highly expressed: {expression[high_mask]}")

# Get gene names for highly expressed genes
high_genes = [gene_names[i] for i in range(len(gene_names)) if high_mask[i]]
print(f"Gene names: {high_genes}")

# Combine conditions: expression between 3 and 10
moderate = (expression >= 3) & (expression <= 10)
print(f"\nModerate expression (3-10): {expression[moderate]}")

In [ ]:
# 2D arrays — gene expression matrices
# Rows = genes, Columns = samples

# 4 genes x 5 samples
expr_matrix = np.array([
    [5.2, 3.8, 7.1, 4.5, 6.3],   # TP53
    [12.4, 15.2, 8.9, 14.1, 11.7], # EGFR
    [3.1, 2.9, 4.2, 3.5, 3.0],    # BRCA1
    [8.7, 9.2, 7.5, 8.1, 8.9],    # KRAS
])

genes = ["TP53", "EGFR", "BRCA1", "KRAS"]
samples = ["S1", "S2", "S3", "S4", "S5"]

print(f"Matrix shape: {expr_matrix.shape} (genes x samples)")
print(f"\nExpression matrix:")
print(f"{'':>8} {' '.join(f'{s:>6}' for s in samples)}")
for gene, row in zip(genes, expr_matrix):
    print(f"{gene:>8} {' '.join(f'{v:>6.1f}' for v in row)}")

# Per-gene statistics (axis=1 means across columns/samples)
print(f"\nPer-gene averages: {np.mean(expr_matrix, axis=1).round(2)}")

# Per-sample statistics (axis=0 means across rows/genes)
print(f"Per-sample averages: {np.mean(expr_matrix, axis=0).round(2)}")

### Try it yourself!
Compute the correlation between TP53 and EGFR expression across the 5 samples.

In [ ]:
# YOUR TURN: Compute correlation between TP53 (row 0) and EGFR (row 1)
tp53 = expr_matrix[0]  # TP53 expression across samples
egfr = expr_matrix[1]  # EGFR expression across samples

# np.corrcoef returns a correlation matrix; [0, 1] gets the correlation between the two
correlation = np.corrcoef(tp53, egfr)[0, 1]
print(f"TP53 values: {tp53}")
print(f"EGFR values: {egfr}")
print(f"Correlation: {correlation:.3f}")

if abs(correlation) > 0.7:
    print("Strong correlation — these genes may be co-regulated!")
elif abs(correlation) > 0.3:
    print("Moderate correlation")
else:
    print("Weak correlation — expression is largely independent")

---

## 2. Pandas — DataFrames for Biology

Pandas is like Excel on steroids. DataFrames have labeled rows and columns, making it intuitive to work with biology datasets. This is the library you'll use most.

In [ ]:
import pandas as pd

# Create a DataFrame from scratch
df = pd.DataFrame({
    'gene': ["TP53", "EGFR", "BRCA1", "KRAS", "MYC", "PTEN", "PIK3CA", "BRAF"],
    'chromosome': ["17p13.1", "7p11.2", "17q21.31", "12p12.1", "8q24.21", "10q23.31", "3q26.32", "7q34"],
    'type': ["tumor_suppressor", "oncogene", "tumor_suppressor", "oncogene", 
             "oncogene", "tumor_suppressor", "oncogene", "oncogene"],
    'expression_tumor': [5.2, 12.4, 3.1, 8.7, 22.1, 4.8, 6.7, 3.2],
    'expression_normal': [6.8, 4.2, 5.5, 3.1, 5.0, 7.2, 3.8, 2.9],
})

# Display the DataFrame
print("Gene Expression: Tumor vs Normal\n")
df

In [ ]:
# Quick overview of your data
print("Shape:", df.shape)  # (rows, columns)
print("\nColumn types:")
print(df.dtypes)
print("\nStatistical summary:")
df.describe()

In [ ]:
# Add calculated columns

# Fold change: how much expression changed in tumor vs normal
df['fold_change'] = df['expression_tumor'] / df['expression_normal']

# Log2 fold change (standard in genomics)
df['log2_fc'] = np.log2(df['fold_change'])

# Is it upregulated or downregulated in tumor?
df['regulation'] = df['log2_fc'].apply(
    lambda x: 'UP' if x > 0.5 else ('DOWN' if x < -0.5 else 'UNCHANGED')
)

print("With calculated columns:\n")
df[['gene', 'type', 'expression_tumor', 'expression_normal', 'log2_fc', 'regulation']]

In [ ]:
# Filtering — select rows based on conditions

# Oncogenes only
oncogenes = df[df['type'] == 'oncogene']
print(f"Oncogenes ({len(oncogenes)}):")
print(oncogenes[['gene', 'expression_tumor', 'log2_fc']].to_string(index=False))

# Upregulated genes
upregulated = df[df['regulation'] == 'UP']
print(f"\nUpregulated in tumor ({len(upregulated)}):")
print(upregulated[['gene', 'log2_fc']].to_string(index=False))

# Combined filter: oncogenes that are upregulated
bad_actors = df[(df['type'] == 'oncogene') & (df['regulation'] == 'UP')]
print(f"\nUpregulated oncogenes ({len(bad_actors)}):")
print(bad_actors[['gene', 'expression_tumor', 'log2_fc']].to_string(index=False))

In [ ]:
# Sorting
print("Genes sorted by log2 fold change (most upregulated first):\n")
sorted_df = df.sort_values('log2_fc', ascending=False)
print(sorted_df[['gene', 'type', 'log2_fc', 'regulation']].to_string(index=False))

In [ ]:
# Grouping — aggregate statistics by category

print("Average expression by gene type:\n")
grouped = df.groupby('type').agg({
    'expression_tumor': ['mean', 'std'],
    'expression_normal': ['mean', 'std'],
    'log2_fc': 'mean',
    'gene': 'count'
}).round(2)

print(grouped)

print("\nRegulation counts by gene type:")
print(pd.crosstab(df['type'], df['regulation']))

---

## 3. Loading Real Data — Breast Cancer Wisconsin Dataset

This is a classic biology ML dataset built into scikit-learn. It contains measurements from digitized images of breast mass cells (569 samples, 30 features per sample).

In [ ]:
from sklearn.datasets import load_breast_cancer

# Load the dataset
cancer = load_breast_cancer()

# Convert to a Pandas DataFrame
df_cancer = pd.DataFrame(cancer.data, columns=cancer.feature_names)
df_cancer['diagnosis'] = cancer.target
df_cancer['diagnosis_label'] = df_cancer['diagnosis'].map({0: 'malignant', 1: 'benign'})

print(f"Dataset shape: {df_cancer.shape}")
print(f"Features: {len(cancer.feature_names)}")
print(f"Samples: {len(df_cancer)}")
print(f"\nDiagnosis distribution:")
print(df_cancer['diagnosis_label'].value_counts())
print(f"\nFirst 5 rows:")
df_cancer.head()

In [ ]:
# Feature overview
print("Feature names (first 10):")
for i, name in enumerate(cancer.feature_names[:10]):
    print(f"  {i+1:2d}. {name}")
print(f"  ... and {len(cancer.feature_names) - 10} more\n")

# Summary statistics for key features
key_features = ['mean radius', 'mean texture', 'mean perimeter', 'mean area', 'mean smoothness']
print("Key feature statistics:\n")
df_cancer[key_features].describe().round(2)

In [ ]:
# Compare features between malignant and benign

print("Mean values by diagnosis:\n")
comparison = df_cancer.groupby('diagnosis_label')[key_features].mean().round(2)
print(comparison)

print("\n\nDifference (malignant - benign):")
diff = comparison.loc['malignant'] - comparison.loc['benign']
for feature, value in diff.items():
    direction = "LARGER" if value > 0 else "smaller"
    print(f"  {feature:<20} {value:>+8.2f}  (malignant is {direction})")

In [ ]:
# Correlation between features

# Compute correlation matrix for key features
corr_matrix = df_cancer[key_features].corr().round(2)

print("Correlation matrix:\n")
print(corr_matrix)

# Find highly correlated pairs
print("\nHighly correlated feature pairs (|r| > 0.8):")
for i in range(len(key_features)):
    for j in range(i+1, len(key_features)):
        r = corr_matrix.iloc[i, j]
        if abs(r) > 0.8:
            print(f"  {key_features[i]} <-> {key_features[j]}: r = {r}")

---

## 4. Handling Missing Data

Real-world biology data almost always has missing values. Pandas represents these as `NaN` (Not a Number) and has powerful tools to deal with them.

In [ ]:
# Create a DataFrame with missing values (simulating a real experiment)
messy_data = pd.DataFrame({
    'patient_id': ['P001', 'P002', 'P003', 'P004', 'P005', 'P006', 'P007', 'P008'],
    'age': [45, 62, 38, np.nan, 55, 71, 43, 59],
    'tumor_size_mm': [22.5, 15.3, np.nan, 31.2, 18.7, np.nan, 12.1, 28.9],
    'TP53_expression': [5.2, 3.8, 7.1, 4.5, np.nan, 2.1, 6.3, 4.8],
    'EGFR_expression': [12.4, np.nan, 8.9, 14.1, 11.7, 13.5, np.nan, 10.2],
    'stage': ['II', 'III', 'I', 'III', 'II', 'IV', 'I', 'III'],
})

print("Messy patient data (NaN = missing):\n")
print(messy_data)

# Check for missing values
print("\nMissing values per column:")
print(messy_data.isnull().sum())
print(f"\nTotal missing: {messy_data.isnull().sum().sum()} out of {messy_data.size} values")

In [ ]:
# Strategy 1: Drop rows with any missing values
clean_v1 = messy_data.dropna()
print(f"After dropping rows with NaN: {len(clean_v1)} rows (lost {len(messy_data) - len(clean_v1)})")

# Strategy 2: Drop rows only if critical columns are missing
clean_v2 = messy_data.dropna(subset=['TP53_expression', 'EGFR_expression'])
print(f"After dropping rows with missing expression: {len(clean_v2)} rows")

# Strategy 3: Fill missing values with the column median (most common for expression data)
clean_v3 = messy_data.copy()
numeric_cols = ['age', 'tumor_size_mm', 'TP53_expression', 'EGFR_expression']
for col in numeric_cols:
    median_val = clean_v3[col].median()
    clean_v3[col] = clean_v3[col].fillna(median_val)
    print(f"  Filled {col} NaN with median = {median_val}")

print(f"\nAfter median imputation: {clean_v3.isnull().sum().sum()} missing values")
print("\nCleaned data:")
clean_v3

---

## 5. Merging Datasets

In biology, you often need to combine data from different sources — expression data with clinical annotations, gene info with pathway memberships, etc.

In [ ]:
# Dataset 1: Gene expression
expression_df = pd.DataFrame({
    'gene_id': ['ENSG001', 'ENSG002', 'ENSG003', 'ENSG004', 'ENSG005'],
    'tumor_expr': [5.2, 12.4, 3.1, 8.7, 22.1],
    'normal_expr': [6.8, 4.2, 5.5, 3.1, 5.0],
})

# Dataset 2: Gene annotations
annotation_df = pd.DataFrame({
    'gene_id': ['ENSG001', 'ENSG002', 'ENSG003', 'ENSG004', 'ENSG005'],
    'gene_name': ['TP53', 'EGFR', 'BRCA1', 'KRAS', 'MYC'],
    'pathway': ['Apoptosis', 'MAPK signaling', 'DNA repair', 'RAS signaling', 'Cell cycle'],
    'druggable': [False, True, False, True, False],
})

# Merge on gene_id (like a VLOOKUP in Excel, but much more powerful)
merged = pd.merge(expression_df, annotation_df, on='gene_id')

# Add fold change
merged['log2_fc'] = np.log2(merged['tumor_expr'] / merged['normal_expr']).round(2)

print("Merged dataset:\n")
print(merged[['gene_name', 'pathway', 'tumor_expr', 'normal_expr', 'log2_fc', 'druggable']].to_string(index=False))

# Quick insight: which druggable targets are upregulated?
targets = merged[(merged['druggable'] == True) & (merged['log2_fc'] > 0)]
print(f"\nDruggable upregulated targets: {list(targets['gene_name'])}")

---

## 6. Saving and Loading DataFrames

Save your results so you can share them or continue later.

In [ ]:
# Save to CSV
merged.to_csv('merged_gene_analysis.csv', index=False)
print("Saved: merged_gene_analysis.csv")

# Load it back
loaded = pd.read_csv('merged_gene_analysis.csv')
print(f"\nLoaded back: {loaded.shape[0]} rows, {loaded.shape[1]} columns")
print(loaded.head())

# Other formats Pandas supports:
# pd.read_excel('file.xlsx')    — Excel files
# pd.read_csv('file.tsv', sep='\t')  — Tab-separated (common in bioinformatics)
# pd.read_json('file.json')     — JSON files
# df.to_excel('output.xlsx')    — Write to Excel

---

## 7. MILESTONE: Breast Cancer Dataset Deep Dive

Put everything together: load the dataset, compute summary statistics per diagnosis class, identify the most discriminative features, and export a clean report.

In [ ]:
from sklearn.datasets import load_breast_cancer
import numpy as np
import pandas as pd

# Load and prepare
cancer = load_breast_cancer()
df = pd.DataFrame(cancer.data, columns=cancer.feature_names)
df['diagnosis'] = cancer.target
df['label'] = df['diagnosis'].map({0: 'malignant', 1: 'benign'})

print(f"Dataset: {df.shape[0]} samples, {df.shape[1] - 2} features")
print(f"Classes: {dict(df['label'].value_counts())}")

# Per-class statistics for all 30 features
class_means = df.groupby('label')[cancer.feature_names].mean()

# Calculate effect size: how different is each feature between classes?
# (difference in means / pooled std — a simplified Cohen's d)
effect_sizes = []

for feature in cancer.feature_names:
    mal = df[df['label'] == 'malignant'][feature]
    ben = df[df['label'] == 'benign'][feature]
    
    pooled_std = np.sqrt((mal.std()**2 + ben.std()**2) / 2)
    if pooled_std > 0:
        d = abs(mal.mean() - ben.mean()) / pooled_std
    else:
        d = 0
    
    effect_sizes.append({
        'feature': feature,
        'malignant_mean': round(mal.mean(), 3),
        'benign_mean': round(ben.mean(), 3),
        'effect_size': round(d, 3),
    })

# Create results DataFrame and sort by effect size
results_df = pd.DataFrame(effect_sizes).sort_values('effect_size', ascending=False)

print("\nTop 10 Most Discriminative Features:\n")
print(f"{'Feature':<25} {'Malignant':>12} {'Benign':>12} {'Effect Size':>12}")
print("=" * 63)
for _, row in results_df.head(10).iterrows():
    print(f"{row['feature']:<25} {row['malignant_mean']:>12.3f} {row['benign_mean']:>12.3f} {row['effect_size']:>12.3f}")

In [ ]:
# Save the complete analysis
results_df.to_csv('breast_cancer_feature_analysis.csv', index=False)
print("Saved: breast_cancer_feature_analysis.csv")
print(f"Contains {len(results_df)} features ranked by effect size.")

# Also save the cleaned dataset for next week's visualization
df.to_csv('breast_cancer_clean.csv', index=False)
print("Saved: breast_cancer_clean.csv")

print("\n" + "=" * 50)
print("  MILESTONE COMPLETE!")
print("=" * 50)
print(f"  Loaded the Breast Cancer Wisconsin dataset")
print(f"  Computed per-class statistics for {len(cancer.feature_names)} features")
print(f"  Top feature: {results_df.iloc[0]['feature']} (effect size = {results_df.iloc[0]['effect_size']})")
print(f"  Exported clean DataFrames to CSV")
print("=" * 50)

---

## Pandas Cheat Sheet

Keep this handy as you work through future weeks!

| Task | Code |
|------|------|
| Load CSV | `df = pd.read_csv('file.csv')` |
| First 5 rows | `df.head()` |
| Shape | `df.shape` |
| Column types | `df.dtypes` |
| Statistics | `df.describe()` |
| Select column | `df['column_name']` |
| Filter rows | `df[df['col'] > 5]` |
| Sort | `df.sort_values('col', ascending=False)` |
| Group & aggregate | `df.groupby('col').mean()` |
| Missing values | `df.isnull().sum()` |
| Fill missing | `df['col'].fillna(df['col'].median())` |
| New column | `df['new'] = df['a'] / df['b']` |
| Merge | `pd.merge(df1, df2, on='key')` |
| Save | `df.to_csv('output.csv', index=False)` |

---

## Week 3 Complete!

**What you learned:**
- NumPy arrays for fast vectorized operations on numerical biology data
- Pandas DataFrames for loading, filtering, grouping, and merging tabular data
- Handling missing values with three different strategies
- Working with the Breast Cancer Wisconsin dataset (your first real ML dataset!)
- Computing effect sizes to identify the most discriminative features

**Next week:** Data visualization with Matplotlib and Seaborn — you'll turn all this data into publication-quality figures including heatmaps, volcano plots, and multi-panel figures.

**Pro tip:** The `breast_cancer_clean.csv` you saved will be used in Week 4 for visualization. Keep it in the same folder!